# Module 05-2: Few-shot 프롬프팅

## 학습 목표
- Zero-shot, One-shot, Few-shot 기법의 차이를 이해할 수 있다
- `ChatPromptTemplate`에 Human/Assistant 쌍으로 Few-shot 예시를 추가할 수 있다
- FakeLLM으로 이메일 분류 체인을 테스트할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/05-prompt-engineering.md` 섹션 2.2

## 개념 요약

### Shot 기법 비교

| 기법 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **Zero-shot** | 예시 없이 지시만 제공 | 간결, 토큰 절약 | 출력 형식 불안정 |
| **One-shot** | 예시 1개 제공 | 형식 이해 도움 | 예시가 편향될 수 있음 |
| **Few-shot** | 예시 2~5개 제공 | 안정적 출력, 높은 품질 | 토큰 사용 증가 |

### Few-shot 메시지 순서

```
[System]    역할 + 규칙           -- 항상 첫 번째
[Human]     Few-shot 입력 1       -- 예시 쌍 시작
[Assistant] Few-shot 출력 1
[Human]     Few-shot 입력 2       -- 필요한 만큼 반복
[Assistant] Few-shot 출력 2
[Human]     실제 입력              -- 항상 마지막
```

Few-shot은 JSON 형식 준수율을 약 **70% → 95%** 이상으로 향상시킵니다.

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from common.fake_llm import FakeLLM

print("환경 설정 완료")

## 실습 1: Zero-shot 이메일 분류기

예시 없이 지시만으로 이메일을 분류하는 Zero-shot 프롬프트를 만들어봅니다.

In [ ]:
# 1-1. Zero-shot 프롬프트 구성
zero_shot = ChatPromptTemplate.from_messages([
    ("system", "이메일을 분류하세요. 카테고리: 문의, 불만, 칭찬, 기타\n반드시 JSON 형식으로만 응답하세요."),
    ("human", "이메일 내용: {email}\n\n카테고리를 JSON으로 답하세요."),
])

# 1-2. FakeLLM으로 테스트 (Zero-shot은 형식 불안정할 수 있음을 시뮬레이션)
zero_shot_llm = FakeLLM(responses={
    "환불": '이메일을 분석하겠습니다. 이 이메일은 문의 카테고리에 해당합니다.',  # JSON이 아닌 자유 텍스트 (형식 불안정 시뮬레이션)
    "배송": '{"category": "불만"}',
})

zero_shot_chain = zero_shot | zero_shot_llm | StrOutputParser()

# 1-3. 테스트
test_emails = [
    "환불 절차가 어떻게 되나요?",
    "배송이 2주나 지연되어 정말 화가 납니다.",
]

print("=== Zero-shot 결과 ===")
for email in test_emails:
    result = zero_shot_chain.invoke({"email": email})
    print(f"이메일: {email[:30]}...")
    print(f"결과: {result}")
    print("-" * 40)

In [ ]:
# 검증 셀
assert zero_shot is not None, "zero_shot 프롬프트가 None입니다."
test_msgs = zero_shot.format_messages(email="테스트 이메일")
assert len(test_msgs) == 2, "메시지가 2개(system + human)이어야 합니다."
assert test_msgs[0].type == "system", "첫 번째 메시지는 system이어야 합니다."
print("실습 1 완료!")

## 실습 2: Few-shot 이메일 분류기

3개의 예시를 포함한 Few-shot 프롬프트를 구성합니다.
Human/Assistant 쌍으로 예시를 추가하는 방법을 학습합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): from_messages()에 System 이후 Human/Assistant 쌍을 반복해서 추가하세요.
#               마지막에 실제 입력을 받는 Human 메시지를 추가합니다.
#
# 힌트 2 (키워드): ("human", "이메일 내용: ..."), ("assistant", '{"category": "문의"}')
#
# 힌트 3 (거의 정답):
#   few_shot = ChatPromptTemplate.from_messages([
#       ("system", "이메일을 분류하세요. 카테고리: 문의, 불만, 칭찬, 기타\n반드시 JSON 형식으로만 응답하세요."),
#       ("human", "이메일 내용: 환불 절차가 어떻게 되나요?"),
#       ("assistant", '{"category": "문의"}'),
#       ("human", "이메일 내용: 배송이 2주나 지연되어 정말 화가 납니다."),
#       ("assistant", '{"category": "불만"}'),
#       ("human", "이메일 내용: 친절한 상담 감사합니다. 문제 해결했어요!"),
#       ("assistant", '{"category": "칭찬"}'),
#       ("human", "이메일 내용: {email}"),
#   ])

# 2-1. Few-shot 프롬프트 구성 (예시 3개 포함)
few_shot = None  # TODO: ChatPromptTemplate.from_messages([...])

# 2-2. FakeLLM 설정 (Few-shot으로 형식이 안정됨을 시뮬레이션)
few_shot_llm = FakeLLM(responses={
    "환불": '{"category": "문의"}',
    "배송": '{"category": "불만"}',
    "감사": '{"category": "칭찬"}',
    "default": '{"category": "기타"}',
})

# 2-3. 체인 구성 및 실행
few_shot_chain = None  # TODO: few_shot | few_shot_llm | StrOutputParser()

if few_shot_chain:
    print("=== Few-shot 결과 ===")
    test_emails = [
        "환불 절차가 어떻게 되나요?",
        "배송이 2주나 지연되어 정말 화가 납니다.",
        "친절한 상담 감사합니다!",
    ]
    for email in test_emails:
        result = few_shot_chain.invoke({"email": email})
        print(f"이메일: {email}")
        print(f"결과: {result}")
        print("-" * 40)

In [ ]:
# 검증 셀
assert few_shot is not None, "few_shot 프롬프트가 None입니다."
assert few_shot_chain is not None, "few_shot_chain이 None입니다."

# 메시지 수 확인: system(1) + 예시3쌍(6) + 실제입력(1) = 8개
msgs = few_shot.format_messages(email="테스트")
assert len(msgs) == 8, f"메시지 수가 8개이어야 합니다 (system + 예시3쌍 + 실제입력). 현재: {len(msgs)}개"
assert msgs[0].type == "system", "첫 번째 메시지는 system이어야 합니다."
assert msgs[-1].type == "human", "마지막 메시지는 human이어야 합니다."
assert msgs[1].type == "human", "두 번째 메시지는 human(few-shot 예시 입력)이어야 합니다."
assert msgs[2].type == "ai", "세 번째 메시지는 ai(few-shot 예시 출력)이어야 합니다."
print("실습 2 완료!")

## 실습 3: Zero-shot vs Few-shot 메시지 구조 비교

두 방법의 메시지 수와 구조를 비교하여 차이를 이해합니다.

In [ ]:
# 3-1. 두 프롬프트의 메시지 수 비교
zero_msgs = zero_shot.format_messages(email="테스트")
few_msgs = few_shot.format_messages(email="테스트")

print("=== Zero-shot 메시지 구조 ===")
for i, msg in enumerate(zero_msgs):
    print(f"  [{i+1}] {msg.type}: {msg.content[:50]}...")

print(f"\n총 {len(zero_msgs)}개 메시지")

print("\n=== Few-shot 메시지 구조 ===")
for i, msg in enumerate(few_msgs):
    content_preview = msg.content[:50] if msg.content else ""
    print(f"  [{i+1}] {msg.type}: {content_preview}...")

print(f"\n총 {len(few_msgs)}개 메시지")

print(f"\n=== 비교 요약 ===")
print(f"Zero-shot: {len(zero_msgs)}개 메시지")
print(f"Few-shot: {len(few_msgs)}개 메시지 ({len(few_msgs) - len(zero_msgs)}개 더 많음)")

In [ ]:
# 검증 셀
assert len(few_msgs) > len(zero_msgs), "Few-shot 메시지가 Zero-shot보다 많아야 합니다."
assert len(few_msgs) - len(zero_msgs) == 6, "3쌍의 예시로 인해 6개 메시지가 추가되어야 합니다."
print("실습 3 완료!")

## 정리

이번 실습에서 학습한 내용:

1. **Zero-shot**: 예시 없이 지시만으로 분류. 간결하지만 형식이 불안정할 수 있음
2. **Few-shot**: Human/Assistant 쌍으로 예시 제공. 형식 준수율 대폭 향상
3. **메시지 순서**: System → (Human+Assistant 쌍 반복) → Human(실제 입력)
4. **트레이드오프**: Few-shot은 토큰을 더 사용하지만 출력 품질이 향상됨

**다음 실습**: `03_yaml_prompts.ipynb` - YAML 파일에서 프롬프트 로드